# H-001 · S2 Universes A–F Discovery + Baseline Backtest

Screens every registered universe and picks `UNIVERSE_STAR`.

| Universe | What | Status |
|----------|------|--------|
| A | FX majors (OANDA) | failed previously — 0 EG passers |
| B | Crypto majors (Kraken) | failed previously — best pair HL > `Z_WINDOW` |
| C | Asia EM cash equities (IBKR HK/JP) | **shelved** — gross Sharpe ≈ 0, net negative after 271–439 bps/yr |
| D | US share-class twins (Alpaca) | candidate |
| E | US REIT sub-sectors (Alpaca) | candidate |
| F | EUR large caps (IBKR EUR) | candidate |

A / B / C are re-run **for the record only** and are documented learning points in
`02_research/s2_coint/universe.md`. They are not re-selected. D / E / F compete for
`UNIVERSE_STAR`.

## Workflow

1. Pool-scoped candidate screening via `iter_pool_pairs` at each universe's fixed `RESEARCH_IS_END`
2. **Deterministic ranked selection** under caps (global 6, per-pool 2) — the old manual
   `KEEP_PAIR_IDS` gate is gone, because a discretionary keep-list is a selection channel we cannot
   audit
3. Export `s2_pairs_{letter}_1d.csv` (ranked book) and `s2_candidates_{letter}_1d.csv` (every
   candidate, so non-selected pairs stay auditable)
4. Research-IS trad-z diagnostics: per-pair trades, cost drag, Sharpe / DD, rolling ADF
5. Book-level gross vs net Sharpe and correlation to S1

Candidates come **only** from within a leaf pool (`S2_POOLS` in
`01_data/processing/s2_universe_pools.py`), at any nesting depth.

## Conventions

- **Research IS only** (`date <= RESEARCH_IS_END`). Do not score sealed OOS here.
- Scored under the `freeze` convention with **soft defaults** (`ols=252`, `z=60`, `entry_z=2`),
  since H-003 (`BREAK_STAR`) and H-004 (`BOOK_STAR`) have not run yet — so **no break gate**.
- Selection ranks on **p-value only**. No half-life filter anywhere before H-008.
- Rank universes on **net Sharpe** and read correlation beside it. **A negative correlation to S1
  is strictly better than a low positive one** (−0.2 beats 0.0), because it adds diversification
  rather than merely avoiding overlap. No combined score.
- Short-selling bans are enforced for F via `strategies.s2_coint.short_bans`; A–E have no records.
- **F cost caveat:** `F_EUR_IBKR` is an assumption pending IBKR's European cash-equity schedule.
  Confirm it before treating F's net Sharpe as decision-grade.

**Run order:** run this notebook **first**, then
`01_data/data_files/s2_coint/s2_pair_panel.ipynb` to build the panels.

## 0. Imports & Config

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.diagnosis import gross_returns_from_net
from backtest.s2_coint.research import candidates_path, pairs_path
from backtest.s2_coint.rotation import simulate_frozen_book
from data.ingestion.equity_fetcher import fetch_ohlcv
from data.processing.feature_implementation.cointegration import COINT_PVALUE
from data.processing.s2_coint_store import build_pair_panel, screen_pair_cointegration
from data.processing.s2_universe import (
    RESEARCH_IS_END_BY_UNIVERSE,
    SHELVED_UNIVERSES,
    iter_pool_pairs,
    load_s2_pools,
    pool_of_pair,
    pool_tickers,
    ticker_venue_key,
)
from strategies.s2_coint.baseline import (
    BETA_COLUMN,
    ENTRY_Z,
    EXIT_Z,
    USE_HEDGE_RATIO_SIZING,
    clip_ohlc_to_is,
)
from strategies.s2_coint.book import CAP_GLOBAL, CAP_PER_POOL, select_book
from strategies.s2_coint.config import S2SimConfig
from strategies.s2_coint.metrics import (
    corr_to_s1,
    diagnose_locked_panel,
    load_s1_period_returns,
    metrics_from_returns,
)

DATA_DIR = os.path.join(ROOT, "01_data", "data_files", "s2_coint")
S1_RETURNS_PATH = os.path.join(
    ROOT, "01_data", "data_files", "s1_equities", "s1_period_returns.parquet"
)

UNIVERSE_LABELS = ("A", "B", "C", "D", "E", "F")
BAR = "1d"
FETCH_START = "1990-01-01"
END_DATE = None
RESEARCH_IS_END = dict(RESEARCH_IS_END_BY_UNIVERSE)

COINT_PVALUE_THRESHOLD = 0.05
OLS_WINDOW = 252
Z_WINDOW = 60
HL_WINDOW = 252
ADF_PVALUE_THRESHOLD = COINT_PVALUE

SIM_KW = dict(
    entry_z=ENTRY_Z,
    exit_z=EXIT_Z,
    beta_column=BETA_COLUMN,
    use_hedge_ratio_sizing=USE_HEDGE_RATIO_SIZING,
)
# Soft defaults: no break gate, since BREAK_STAR is not frozen until H-003.
BOOK_CFG = S2SimConfig(
    bar=BAR,
    entry_z=ENTRY_Z,
    exit_z=EXIT_Z,
    ols_window=OLS_WINDOW,
    z_window=Z_WINDOW,
    hl_window=HL_WINDOW,
    adf_window=OLS_WINDOW,
    break_mode="off",
)

POOLS = {label: load_s2_pools(label) for label in UNIVERSE_LABELS}
CANDIDATES = {label: iter_pool_pairs(POOLS[label]) for label in UNIVERSE_LABELS}
POOL_OF_PAIR = {label: pool_of_pair(POOLS[label]) for label in UNIVERSE_LABELS}

for label in UNIVERSE_LABELS:
    flag = "  (shelved - record only)" if label in SHELVED_UNIVERSES else ""
    print(
        f"{label}: {len(pool_tickers(POOLS[label])):3d} tickers  "
        f"{len(CANDIDATES[label]):3d} candidate pairs  IS end {RESEARCH_IS_END[label]}{flag}"
    )
print("\ntotal candidates:", sum(len(v) for v in CANDIDATES.values()))
print("caps: global", CAP_GLOBAL, "| per pool", CAP_PER_POOL)

## 1. Load pools and fetch OHLCV

`isAsian=True` keeps the exchange-suffix dot for `.HK` / `.T` / `.MC` / `.MI` / `.AS` / `.DE` /
`.PA`. US names, **including share classes such as `BF.B`**, must stay `isAsian=False` so the fetcher
hyphenates them to Yahoo's `BF-B`.

In [ ]:
ohlc_by_universe: dict[str, dict[str, pd.DataFrame]] = {}
closes_by_universe: dict[str, dict[str, pd.Series]] = {}

for label in UNIVERSE_LABELS:
    ohlc: dict[str, pd.DataFrame] = {}
    closes: dict[str, pd.Series] = {}
    print(f"\n=== Universe {label} ===")
    for ticker in pool_tickers(POOLS[label]):
        is_suffixed = "." in ticker and ticker_venue_key(ticker) != "US"
        panel = fetch_ohlcv(
            ticker,
            start_date=FETCH_START,
            end_date=END_DATE,
            isAsian=is_suffixed,
            interval=BAR,
        )
        if panel is None or panel.empty:
            print(f"WARNING: no OHLCV for {ticker} ({label})")
            continue
        frame = (
            panel.sort_values("date")
            .drop_duplicates("date", keep="last")
            .set_index("date")[["open", "high", "low", "close"]]
            .astype(float)
        )
        frame.index = pd.to_datetime(frame.index)
        frame = frame.dropna(subset=["close"])
        if frame.empty:
            print(f"WARNING: no valid closes for {ticker} ({label})")
            continue
        ohlc[ticker] = frame
        closes[ticker] = frame["close"].copy()
        print(
            f"{ticker:12} bars={len(frame):5d} venue={ticker_venue_key(ticker)} "
            f"[{frame.index.min().date()} .. {frame.index.max().date()}]"
        )
    ohlc_by_universe[label] = ohlc
    closes_by_universe[label] = closes

## 2. Research IS cutoff

Pre-registered calendar `RESEARCH_IS_END` per universe (no fraction of dates). Screen and train both
use `date <= T`. D / E / F are all `2021-12-31`, so cross-universe comparison is apples-to-apples
with ~14 quarters of sealed OOS left for H-004's rotating book.

In [ ]:
is_end_by_universe: dict[str, pd.Timestamp] = {}
for label in UNIVERSE_LABELS:
    is_end_by_universe[label] = pd.Timestamp(RESEARCH_IS_END[label])
    print(f"{label}: RESEARCH_IS_END={is_end_by_universe[label].date()}")

## 3. Pool-scoped screening (fixed `RESEARCH_IS_END`)

Engle-Granger on `date <= T` for candidates **within leaf pools only**. `eligible=False` means too
few mutual IS bars — that is a data gap, not an EG failure. The screen tests both `y~x` and `x~y`
and orients `pair_id` by the lower p-value.

In [ ]:
screened_by_universe: dict[str, pd.DataFrame] = {}

for label in UNIVERSE_LABELS:
    closes = closes_by_universe[label]
    candidates = [
        (a, b) for a, b in CANDIDATES[label] if a in closes and b in closes
    ]
    print(f"\n=== Universe {label}: {len(candidates)} screenable candidates ===")
    screened = screen_pair_cointegration(
        closes,
        candidates,
        is_end=is_end_by_universe[label],
        ols_window=OLS_WINDOW,
        pvalue_threshold=COINT_PVALUE_THRESHOLD,
    )
    if not screened.empty:
        screened["pool"] = [
            POOL_OF_PAIR[label].get(str(p), "?") for p in screened["pair_id"]
        ]
    screened_by_universe[label] = screened
    if screened.empty:
        print("  no screenable candidates")
        continue
    n_elig = int(screened["eligible"].sum())
    n_pass = int(
        (screened["eligible"] & (screened["pvalue"] < COINT_PVALUE_THRESHOLD)).sum()
    )
    print(f"  eligible {n_elig}/{len(screened)}  EG passers {n_pass}")
    display(
        screened.sort_values("pvalue")[
            ["pair_id", "pool", "pvalue", "discovery_half_life", "n_is_bars", "eligible"]
        ].head(12)
    )

## 4. Ranked selection under caps → export artifacts

Replaces the manual keep gate. `select_book` drops ineligible rows and α failures, ranks the rest by
p-value, then applies the **per-pool cap (2)** before the **global cap (6)**. An unordered pair
occupies one slot regardless of orientation.

The per-pool cap is what stops a single factor filling the book — the failure mode that sank C, where
all three locked pairs loaded one China-bank factor.

In [ ]:
selected_by_universe: dict[str, pd.DataFrame] = {}
selected_pairs_by_universe: dict[str, list[tuple[str, str]]] = {}

for label in UNIVERSE_LABELS:
    screened = screened_by_universe[label]
    chosen = select_book(
        screened,
        pool_of_pair=POOL_OF_PAIR[label],
        per_pool_cap=CAP_PER_POOL,
        global_cap=CAP_GLOBAL,
        pvalue_threshold=COINT_PVALUE_THRESHOLD,
    )
    selected_by_universe[label] = chosen
    selected_pairs_by_universe[label] = [
        (str(r.ticker_y), str(r.ticker_x)) for r in chosen.itertuples(index=False)
    ]

    # Every candidate stays auditable, not just the selected ones.
    if not screened.empty:
        screened.to_csv(candidates_path(BAR, universe=label, root=ROOT), index=False)
    if not chosen.empty:
        chosen.to_csv(pairs_path(BAR, universe=label, root=ROOT), index=False)

    print(f"{label}: selected {len(chosen)} pairs")
    if not chosen.empty:
        print("   by pool:", chosen["pool"].value_counts().to_dict())
        display(chosen[["rank", "pair_id", "pool", "pvalue", "discovery_half_life"]])

## 5. Build selected-pair research-IS panels

OHLC is clipped to `RESEARCH_IS_END` **before** the panel is built, so no OOS bar can leak into a
rolling window.

In [ ]:
is_panels: dict[str, pd.DataFrame] = {}

for label in UNIVERSE_LABELS:
    pairs = selected_pairs_by_universe[label]
    if not pairs:
        print(f"{label}: no selected pairs - skipping panel")
        is_panels[label] = pd.DataFrame()
        continue
    clipped = clip_ohlc_to_is(ohlc_by_universe[label], is_end_by_universe[label])
    panel = build_pair_panel(
        clipped,
        pairs,
        ols_window=OLS_WINDOW,
        z_window=Z_WINDOW,
        hl_window=HL_WINDOW,
        include_adf_pvalue=True,
    )
    is_panels[label] = panel
    print(
        f"{label}: panel shape={panel.shape} pairs={panel['pair_id'].nunique()} "
        f"[{panel['date'].min().date()} .. {panel['date'].max().date()}]"
    )

## 6. Per-pair IS diagnostics

Descriptive per-pair table: trade count, median hold, cost bps/year, Sharpe / max DD and rolling
ADF. Note this table runs the plain baseline simulator, so it does **not** apply short-ban masks —
the book-level numbers in section 8 do. For F, treat these per-pair Sharpes as slightly optimistic.

In [ ]:
pair_diag_by_universe: dict[str, pd.DataFrame] = {}

for label in UNIVERSE_LABELS:
    panel = is_panels[label]
    if panel.empty:
        continue
    table = diagnose_locked_panel(
        panel, adf_pvalue_threshold=ADF_PVALUE_THRESHOLD, **SIM_KW
    )
    pair_diag_by_universe[label] = table
    print(f"\n=== Universe {label} per-pair IS diagnostics ===")
    display(table)

## 7. Rolling ADF paths (research IS)

Health, not discovery. C's lesson: EG passed once at the IS end while rolling ADF sat above 0.05 for
71–89% of days, so the spread was usually not stationary. Look for a pair that spends most of its
life **below** the threshold line, and for pools whose ADF paths move together (correlated health =
one shared factor, not independent bets).

In [ ]:
for label in UNIVERSE_LABELS:
    panel = is_panels[label]
    if panel.empty or "adf_pvalue" not in panel.columns:
        continue
    pair_ids = sorted(panel["pair_id"].unique())
    fig, axes = plt.subplots(
        len(pair_ids), 1, figsize=(11, 2.2 * len(pair_ids)), sharex=True, squeeze=False
    )
    for ax, pid in zip(axes[:, 0], pair_ids):
        g = panel.loc[panel["pair_id"] == pid].sort_values("date")
        ax.plot(g["date"], g["adf_pvalue"], lw=0.9)
        ax.axhline(ADF_PVALUE_THRESHOLD, ls="--", lw=0.8, color="red")
        pct = float((g["adf_pvalue"] < ADF_PVALUE_THRESHOLD).mean() * 100.0)
        ax.set_title(f"{pid}  ({pct:.0f}% of days p<{ADF_PVALUE_THRESHOLD})", fontsize=9)
        ax.set_ylabel("ADF p")
    fig.suptitle(f"Universe {label} rolling ADF (research IS)", y=1.001)
    fig.tight_layout()
    plt.show()

## 8. Universe IS book summary

Book-level metrics via `simulate_frozen_book`, so short-ban masks and the fixed `1/6` slot weight are
in force. Gross adds fill costs back onto the net series rather than re-simulating.

**How to read it.** `ann_sharpe_gross` ≫ 0 with `ann_sharpe_net` ≤ 0 means costs ate the edge; both
≤ 0 means there was no edge to begin with (that was C). Rank on net Sharpe, and prefer negative
`corr_to_s1` over a small positive one.

In [ ]:
s1_weekly = load_s1_period_returns(S1_RETURNS_PATH)
if s1_weekly.empty:
    print("NOTE: s1_period_returns.parquet missing - corr_to_s1 will be NaN")

rows = []
for label in UNIVERSE_LABELS:
    panel = is_panels[label]
    chosen = selected_by_universe[label]
    if panel.empty or chosen.empty:
        rows.append(
            {
                "universe": label,
                "n_pairs": 0,
                "ann_sharpe_gross": float("nan"),
                "ann_sharpe_net": float("nan"),
                "max_drawdown": float("nan"),
                "corr_to_s1": float("nan"),
                "n_days": 0,
                "shelved": label in SHELVED_UNIVERSES,
            }
        )
        continue
    book = simulate_frozen_book(panel, list(chosen["pair_id"]), BOOK_CFG)
    net = book["returns"]
    gross = gross_returns_from_net(net, book.get("trades", pd.DataFrame()))
    m_net = metrics_from_returns(net)
    rows.append(
        {
            "universe": label,
            "n_pairs": int(len(chosen)),
            "ann_sharpe_gross": metrics_from_returns(gross)["ann_sharpe"],
            "ann_sharpe_net": m_net["ann_sharpe"],
            "max_drawdown": m_net["max_drawdown"],
            "corr_to_s1": corr_to_s1(net, s1_weekly),
            "n_days": m_net["n_days"],
            "shelved": label in SHELVED_UNIVERSES,
        }
    )

book_summary = pd.DataFrame(rows)
print("Universe IS book summary (research IS only):")
display(book_summary)

eligible = book_summary.loc[~book_summary["shelved"]].dropna(subset=["ann_sharpe_net"])
if eligible.empty:
    print("\nNo selectable universe produced a book - nothing to freeze.")
else:
    print("\nSelectable universes ranked by net Sharpe (corr shown, not scored):")
    display(
        eligible.sort_values("ann_sharpe_net", ascending=False)[
            ["universe", "ann_sharpe_net", "ann_sharpe_gross", "max_drawdown", "corr_to_s1"]
        ]
    )

## 9. Freeze `UNIVERSE_STAR`

Pick from **D / E / F only** — A / B / C are shelved and stay documented failures. Weigh net Sharpe
first, then correlation to S1 (negative is better than low positive), then how concentrated the book
is by pool.

Also set `BAR_STAR` in H-002 and `RESEARCH_IS_END_STAR` to the chosen universe's registered end.

In [ ]:
from backtest.s2_coint.report import load_star_stack, require_star, save_star_stack
from backtest.s2_coint.research import DEFAULT_STAR_STACK

UNIVERSE_STAR = None  # TODO: "D" | "E" | "F"
require_star("UNIVERSE_STAR", UNIVERSE_STAR)
if UNIVERSE_STAR in SHELVED_UNIVERSES:
    raise ValueError(
        f"{UNIVERSE_STAR} is shelved (documented failure). Choose from D / E / F."
    )

stack = load_star_stack(DEFAULT_STAR_STACK)
stack["UNIVERSE_STAR"] = str(UNIVERSE_STAR)
stack["RESEARCH_IS_END_STAR"] = RESEARCH_IS_END[str(UNIVERSE_STAR)]
save_star_stack(DEFAULT_STAR_STACK, stack)
print(
    "wrote UNIVERSE_STAR", stack["UNIVERSE_STAR"],
    "RESEARCH_IS_END_STAR", stack["RESEARCH_IS_END_STAR"],
)

## 10. Notes for next hyp

Run `01_data/data_files/s2_coint/s2_pair_panel.ipynb` next with `BOOK_SCOPE = "freeze_only"` to
materialise panels, then H-002 (`BAR_STAR`) → H-003 (`BREAK_STAR`) → H-004 (`BOOK_STAR`) → H-005
(window STARs).

Do not re-pick the universe after seeing downstream bake-offs. If H-003 or H-004 shelves the chosen
universe, come back here and take the next-ranked one — that is a documented step, not a re-pick.